# Paper 08 · Deep Q-Networks

**Citation:** Volodymyr Mnih et al., “Human-level control through deep reinforcement learning” (Nature, 2015).

**Paper:** https://doi.org/10.1038/nature14236

> **Scale gap:** We use a tiny chain-control problem, not Atari pixels. The notebook targets replay, target networks, and TD learning.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 03 · Probability & Bayes](../../math/03_probability_bayes.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 13 · Reinforcement-Learning Mathematics](../../math/13_reinforcement_learning_math.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. Why are consecutive RL samples strongly correlated?
2. What instability can a moving bootstrap target create?
3. Why must evaluation separate exploration from the learned policy?

## Central claim
Deep Q-learning can be stabilized enough to learn control policies by combining neural value approximation with replay and a target network.

## Tiny environment and replay buffer

In [ ]:
import random, numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
from collections import deque
torch.manual_seed(0); random.seed(0); np.random.seed(0)

class Chain:
    def __init__(self,n=7,max_steps=20): self.n=n; self.max_steps=max_steps
    def reset(self): self.s=0; self.steps=0; return self.obs()
    def obs(self):
        x=np.zeros(self.n,np.float32); x[self.s]=1; return x
    def step(self,a):
        self.steps+=1
        self.s=max(0,min(self.n-1,self.s+(1 if a else -1)))
        done=self.s==self.n-1 or self.steps>=self.max_steps
        return self.obs(),(1.0 if self.s==self.n-1 else 0.0),done

class QNet(nn.Module):
    def __init__(self,n): super().__init__(); self.net=nn.Sequential(nn.Linear(n,32),nn.ReLU(),nn.Linear(32,2))
    def forward(self,x): return self.net(x)

## DQN trainer
TODO: after the first run, explain each stabilizer in the marked lines.

In [ ]:
def train_dqn(use_target=True,use_replay=True,episodes=350,seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    env=Chain(); q=QNet(env.n); target=QNet(env.n); target.load_state_dict(q.state_dict())
    opt=torch.optim.Adam(q.parameters(),lr=.01); buf=deque(maxlen=1000); success=[]
    for ep in range(episodes):
        s=env.reset(); eps=max(.05,1-ep/250); won=0
        for step in range(env.max_steps):
            a=random.randrange(2) if random.random()<eps else int(q(torch.tensor(s)[None]).argmax())
            ns,r,d=env.step(a); buf.append((s,a,r,ns,d)); s=ns
            batch=random.sample(buf,min(32,len(buf))) if use_replay else [buf[-1]]
            S=torch.tensor(np.array([x[0] for x in batch])); A=torch.tensor([x[1] for x in batch])[:,None]
            R=torch.tensor([x[2] for x in batch]); NS=torch.tensor(np.array([x[3] for x in batch]))
            D=torch.tensor([x[4] for x in batch],dtype=torch.float32)
            pred=q(S).gather(1,A).squeeze(1)
            boot=(target if use_target else q)(NS).max(1).values.detach()
            y=R+.95*(1-D)*boot
            loss=((pred-y)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
            if use_target and ep%10==0 and step==0: target.load_state_dict(q.state_dict())
            if d: won=int(r>0); break
        success.append(won)
    return q,np.array(success)

## Reproduce stabilization trend

In [ ]:
variants={}
for name,ut,ur in [("DQN",True,True),("no target",False,True),("no replay",True,False)]:
    _,s=train_dqn(ut,ur,seed=1)
    variants[name]=s
    print(name,"last-50 success",s[-50:].mean())
    plt.plot(np.convolve(s,np.ones(30)/30,mode="valid"),label=name)
plt.legend(); plt.ylabel("moving success rate"); plt.xlabel("episode"); plt.show()

### Ablation
Repeat across 5 seeds. A single lucky RL curve is not strong evidence.

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this paper?
2. What was actually new?
3. What evidence did your notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which idea survived into modern systems?
6. What would you test next?